<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Hello, world: your first CuTe DSL kernel

The **CuTe DSL** lets you write a GPU kernel in Python and compile it straight to CUDA. Two
decorators do the work: **`@cute.kernel`** marks the *device* code that every GPU thread runs, and
**`@cute.jit`** marks the *host* function that launches it. You hand the kernel plain PyTorch CUDA
tensors and read the result straight back -- no C++, no separate build step.

This first chapter is deliberately tiny: one kernel that adds two arrays and prints a greeting from
the GPU. Every later chapter builds on this shape.

**You'll learn:** the two decorators -- `@cute.kernel` (device) and `@cute.jit` (host launcher); how
a thread finds its position with `cute.arch.thread_idx()` / `block_idx()` / `block_dim()`; how to
print from the device with `cute.printf`; and the cutlass <-> PyTorch round trip via
`cute.runtime.from_dlpack`.

**Runs on:** any CUDA GPU.

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. The kernel -- `@cute.kernel`

A `@cute.kernel` function is the code **one GPU thread** runs; the GPU runs thousands of copies of it
in parallel. A thread locates its own slot from three built-ins -- `cute.arch.thread_idx()` (lane
within the block), `cute.arch.block_idx()` (which block), and `cute.arch.block_dim()` (block size) --
each a 3-tuple `(x, y, z)`; we use `x`. The global element index is `block * block_size + thread`.

Kernel arguments are **`cutlass.Array`**: a typed handle to GPU memory you index like a Python
sequence. `cute.printf` prints from the device -- it takes a format string with `{}` holes followed
by the values. Here only thread 0 greets us, since thousands of threads printing would flood the
console.

In [ ]:
@cute.kernel
def hello_kernel(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32):
    tx, _, _ = cute.arch.thread_idx()   # lane within the block
    bx, _, _ = cute.arch.block_idx()    # which block
    bdx, _, _ = cute.arch.block_dim()   # threads per block
    i = bx * bdx + tx                   # this thread's global element

    # One thread says hello (every thread printing would flood the console).
    if i == 0:
        cute.printf("hello from the GPU -- block {}, thread {}\n", bx, tx)

    # Each thread adds one element. Guard the tail: the grid rounds up to whole
    # blocks, so the last block can have threads whose index runs past the array.
    if i < N:
        c[i] = a[i] + b[i]

## 2. The launcher -- `@cute.jit`

The `@cute.jit` host function picks the launch shape and starts the kernel with
`.launch(grid=, block=)`: `grid` is the number of blocks, `block` the threads per block. We use one
thread per element and round the grid up so every element is covered.

To call it, wrap each PyTorch CUDA tensor with `cute.runtime.from_dlpack` -- a zero-copy view the DSL
accepts wherever a `cutlass.Array` is expected -- then just call the host function.

In [ ]:
@cute.jit
def hello(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32):
    block = (256, 1, 1)
    grid = ((N + 255) // 256, 1, 1)   # one thread per element, rounded up to whole blocks
    hello_kernel(a, b, c, N).launch(grid=grid, block=block)


N = 1024
a = torch.randn(N, dtype=torch.float32, device="cuda")
b = torch.randn(N, dtype=torch.float32, device="cuda")
c = torch.zeros(N, dtype=torch.float32, device="cuda")

# from_dlpack wraps each CUDA tensor as a cutlass.Array with no copy.
hello(cute.runtime.from_dlpack(a), cute.runtime.from_dlpack(b), cute.runtime.from_dlpack(c), N)

torch.testing.assert_close(c.cpu(), (a + b).cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output (the GPU greeting prints once, from thread 0):
# hello from the GPU -- block 0, thread 0
# PASS

## Try it yourself

- Change `N` and watch the tail guard `if i < N` earn its keep when it is not a multiple of 256.
- Make **every** thread print its index -- then see why the greeting is guarded to thread 0.
- Swap the `+` for a `*`, or add a third input.

Next, `02_control_flow` covers loops and compile-time branching; over in `2_primitives`,
`01_array_concepts` digs into `cutlass.Array` itself -- the memory spaces, multi-dimensional
indexing, and vectorized slices.